# Mini-GPT: Ein trainiertes Modell laden und erkunden

In `02_mini_gpt.ipynb` hast du ein Modell trainiert und in Schritt 8 gespeichert. Hier lädst du so ein **fertig trainiertes** Modell wieder – ganz ohne erneutes Training – und kannst sofort damit Text generieren und hineinschauen.

So kannst du z. B.
- ein Modell weiterverwenden, dessen Training lange gedauert hat (etwa ein großes Modell, das auf einer GPU trainiert wurde),
- Modelle von Mitschüler:innen ausprobieren,
- Modelle vergleichen, die auf verschiedenen Texten oder mit verschiedenen Einstellungen trainiert wurden.

### Ablauf

| Schritt | Was passiert |
|---|---|
| 0 | Setup – GPU oder CPU? |
| 1 | Modell laden |
| 2 | Text generieren |
| 3 | Ins Modell hineinschauen |
| 4 | Modelle vergleichen |

### So arbeitest du mit diesem Notebook
- Führe die Zellen **der Reihe nach** aus: Zelle anklicken → **Shift + Enter**.
- **Aufgabe**-Abschnitte sind für dich: Probiere etwas aus und notiere deine Beobachtungen in den **Deine Antwort**-Zellen darunter.
- Die Zellen **Hilfsfunktionen** und **Modell-Bauplan** enthalten Hilfscode. Den musst du nicht verstehen.
- Wenn etwas hängt oder kaputt ist: **Kernel → Restart Kernel** und die Zellen noch einmal von oben ausführen.

## 0. Setup

Die nächste Zelle prüft, ob eine NVIDIA-GPU verfügbar ist – sonst wird die CPU verwendet. Ein Modell, das auf der GPU trainiert wurde, läuft auch auf der CPU.

In [ ]:
%matplotlib inline
import math
import time
import shutil

import torch
import torch.nn as nn
import torch.nn.functional as F
import ipywidgets as widgets
from IPython.display import display

torch.manual_seed(42)

if torch.cuda.is_available():
    device = "cuda"
    print(f"NVIDIA-GPU gefunden: {torch.cuda.get_device_name(0)} – das Modell läuft auf der GPU.")
else:
    device = "cpu"
    print("Keine nutzbare NVIDIA-GPU gefunden – das Modell läuft auf der CPU.")
    print("   Für dieses Notebook reicht das völlig – hier wird nur Text generiert, nicht trainiert.")
    if shutil.which("nvidia-smi"):
        print("\nAuf diesem Computer ist eine NVIDIA-GPU, aber PyTorch wurde ohne GPU-Unterstützung installiert.")
        print("   Installiere die GPU-Version im Terminal und starte danach den Kernel neu:")
        print("   pip install torch --index-url https://download.pytorch.org/whl/cu128")

In [ ]:
# Hilfsfunktionen zum Laden und für Visualisierungen.
#    Du musst diesen Code nicht verstehen – einfach ausführen!
import html
import io
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, clear_output

_repo_data = Path.cwd().parent / "data"
DATA_DIR = _repo_data if _repo_data.is_dir() else Path.cwd() / "data"
DATA_DIR.mkdir(exist_ok=True)

CATEGORY_COLORS = {
    "Vokal": "#e15759", "Konsonant": "#4e79a7", "Großbuchstabe": "#f28e2b",
    "Ziffer": "#59a14f", "Leerzeichen / Umbruch": "#9c755f", "Satzzeichen / Sonstiges": "#b07aa1",
}


def _fmt(n):
    return f"{n:,}".replace(",", ".")


def _visible(ch):
    return {" ": "␣", "\n": "↵", "\t": "⇥"}.get(ch, ch)


def _highlight(prompt, result):
    return (f"<pre style='background:#f6f8fa;color:#222;padding:10px;white-space:pre-wrap;font-size:13px'>"
            f"<b style='background:#ffe08a'>{html.escape(prompt)}</b>{html.escape(result[len(prompt):])}</pre>")


def generation_playground(generate_fn, encode, decode, default_prompt):
    prompt = widgets.Textarea(value=default_prompt, description="Anfang:", layout=widgets.Layout(width="600px", height="60px"))
    length = widgets.IntSlider(value=300, min=50, max=1500, step=50, description="Länge:")
    temperature = widgets.FloatSlider(value=0.8, min=0.1, max=2.0, step=0.1, description="Temperatur:")
    button = widgets.Button(description="Text generieren", button_style="success")
    out = widgets.Output()

    def on_click(_):
        with out:
            clear_output()
            print("Das Modell schreibt …")
            result = generate_fn(prompt.value, length.value, temperature.value)
            clear_output()
            display(HTML(_highlight(decode(encode(prompt.value)), result)))
    button.on_click(on_click)
    display(widgets.VBox([prompt, length, temperature, button, out]))
    return button


def plot_next_char(model, encode, itos, prompt, temperature, top=15):
    ids = encode(prompt)[-model.block_size:]
    if not ids:
        print("Bitte gib einen Text ein (mit Zeichen, die im Vokabular vorkommen).")
        return
    dev = next(model.parameters()).device
    model.eval()
    with torch.no_grad():
        logits, _ = model(torch.tensor([ids], device=dev))
    model.train()
    probs = F.softmax(logits[0, -1] / temperature, dim=-1).cpu()
    p, idx = probs.topk(min(top, len(probs)))
    labels = [f"„{_visible(itos[i])}“" for i in idx.tolist()]
    fig, ax = plt.subplots(figsize=(8, 4.5))
    bars = ax.barh(labels[::-1], (p * 100).tolist()[::-1], color="#4e79a7")
    ax.bar_label(bars, fmt="%.1f %%", padding=3)
    ax.set_xlim(0, 105)
    ax.set_xlabel("Wahrscheinlichkeit in %")
    shown = prompt[-25:].replace("\n", "↵")
    ax.set_title(f"Welches Zeichen kommt nach „…{shown}“?   (Temperatur {temperature})")
    plt.show()


def next_char_explorer(model, encode, itos, default_prompt):
    prompt = widgets.Text(value=default_prompt, description="Text:", continuous_update=False,
                          layout=widgets.Layout(width="600px"))
    temperature = widgets.FloatSlider(value=1.0, min=0.1, max=2.0, step=0.1, description="Temperatur:",
                                      continuous_update=False)
    out = widgets.interactive_output(
        lambda p, t: plot_next_char(model, encode, itos, p, t), {"p": prompt, "t": temperature})
    display(widgets.VBox([prompt, temperature]), out)


def plot_attention(model, encode, itos, text, layer, head, max_len=40):
    ids = encode(text)[-min(max_len, model.block_size):]
    if len(ids) < 2:
        print("Bitte gib mindestens 2 Zeichen ein.")
        return
    dev = next(model.parameters()).device
    model.eval()
    with torch.no_grad():
        model(torch.tensor([ids], device=dev))
    model.train()
    heads = model.blocks[layer].sa.heads
    labels = [_visible(itos[i]) for i in ids]
    selected = range(len(heads)) if head == -1 else [head]
    cols = min(len(selected), 3)
    rows = math.ceil(len(selected) / cols)
    size = 7 if head != -1 else 4.5
    fig, axes = plt.subplots(rows, cols, figsize=(size * cols, size * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, h in zip(axes.flat, selected):
        ax.axis("on")
        ax.imshow(heads[h].last_attention[0].cpu(), cmap="Blues", vmin=0, vmax=1)
        ax.set_xticks(range(len(labels)), labels, fontsize=8)
        ax.set_yticks(range(len(labels)), labels, fontsize=8)
        ax.set_title(f"Block {layer + 1}, Kopf {h + 1}")
        if head != -1:
            ax.set_xlabel("… schaut auf dieses Zeichen")
            ax.set_ylabel("Dieses Zeichen …")
    fig.tight_layout()
    plt.show()


def attention_explorer(model, encode, itos, default_text):
    n_layer, n_head = len(model.blocks), len(model.blocks[0].sa.heads)
    text = widgets.Text(value=default_text, description="Text:", continuous_update=False,
                        layout=widgets.Layout(width="600px"))
    layer = widgets.IntSlider(value=1, min=1, max=n_layer, description="Block:", continuous_update=False)
    head = widgets.Dropdown(options=[("Alle Köpfe", -1)] + [(f"Kopf {i + 1}", i) for i in range(n_head)],
                            value=0, description="Kopf:")
    out = widgets.interactive_output(
        lambda t, l, h: plot_attention(model, encode, itos, t, l - 1, h), {"t": text, "l": layer, "h": head})
    display(widgets.VBox([text, widgets.HBox([layer, head])]), out)


def _category(ch):
    if ch.lower() in "aeiouäöüy" and ch.islower():
        return "Vokal"
    if ch.isalpha():
        return "Großbuchstabe" if ch.isupper() else "Konsonant"
    if ch.isdigit():
        return "Ziffer"
    if ch.isspace():
        return "Leerzeichen / Umbruch"
    return "Satzzeichen / Sonstiges"


def plot_embeddings(model, itos):
    weights = model.token_embedding.weight.detach().cpu().float().numpy()
    weights = weights - weights.mean(axis=0)
    _, _, vt = np.linalg.svd(weights, full_matrices=False)
    points = weights @ vt[:2].T
    fig, ax = plt.subplots(figsize=(10, 8))
    for i, (x, y) in enumerate(points):
        color = CATEGORY_COLORS[_category(itos[i])]
        ax.scatter(x, y, color=color, s=10)
        ax.annotate(_visible(itos[i]), (x, y), fontsize=12, color=color, ha="center", va="bottom")
    for name, color in CATEGORY_COLORS.items():
        ax.scatter([], [], color=color, label=name)
    ax.legend(loc="best")
    ax.set_title("Wo liegen die Zeichen im Embedding-Raum? (auf 2 Dimensionen reduziert)")
    ax.grid(alpha=0.3)
    plt.show()


CHECKPOINT_KEYS = {"model", "config", "chars", "history"}


def _load_checkpoint(source, device):
    checkpoint = torch.load(source, map_location=device, weights_only=True)
    if not isinstance(checkpoint, dict) or not CHECKPOINT_KEYS <= checkpoint.keys():
        raise ValueError("Diese Datei ist kein Mini-GPT-Modell. Speichere ein Modell in Schritt 8 von "
                         "02_mini_gpt.ipynb und lade es dann hier.")
    return checkpoint


class ModelChooser:
    SOURCES = ["Modell aus data/-Ordner", "Datei hochladen"]

    def __init__(self, device):
        self.device = device
        self.source = widgets.ToggleButtons(options=self.SOURCES, description="Quelle:")
        files = sorted(DATA_DIR.glob("*.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
        options = [(f"{p.name}  ({p.stat().st_size / 1e6:.1f} MB)", p.name) for p in files]
        self.local = widgets.Dropdown(options=options or [("(keine .pt-Dateien gefunden)", "")],
                                      description="Modell:", layout=widgets.Layout(width="480px"))
        self.upload = widgets.FileUpload(accept=".pt", multiple=False, description="Datei wählen")
        self.pages = {
            "Modell aus data/-Ordner": widgets.VBox([
                widgets.HTML(f"Modelle aus Schritt 8 von <code>02_mini_gpt.ipynb</code> liegen im Ordner "
                             f"<code>{html.escape(str(DATA_DIR))}</code>. Neue Dateien erscheinen, "
                             "wenn du diese Zelle erneut ausführst."), self.local]),
            "Datei hochladen": widgets.VBox([
                widgets.HTML("Wähle eine <b>.pt</b>-Datei von deinem Computer aus. Große Modelle (über ca. 10 MB) "
                             "lassen sich evtl. nicht hochladen – lege sie dann in den <code>data/</code>-Ordner."),
                self.upload]),
        }
        self.box = widgets.VBox()
        self.source.observe(lambda _: self._refresh(), names="value")
        self._refresh()

    def _refresh(self):
        self.box.children = [self.source, self.pages[self.source.value]]

    def show(self):
        display(self.box)

    def load(self):
        if self.source.value == "Modell aus data/-Ordner":
            if not self.local.value:
                raise ValueError(f"Keine Datei ausgewählt. Lege eine .pt-Datei in {DATA_DIR} und führe die Zelle oben erneut aus.")
            path = DATA_DIR / self.local.value
            return path.name, _load_checkpoint(path, self.device)
        if not self.upload.value:
            raise ValueError("Du hast noch keine Datei hochgeladen. Wähle oben eine Datei aus und führe diese Zelle erneut aus.")
        file = self.upload.value[0]
        return file["name"], _load_checkpoint(io.BytesIO(bytes(file["content"])), self.device)


CONFIG_LABELS = {
    "block_size": "Kontextlänge", "n_embd": "Embedding-Größe", "n_head": "Attention-Köpfe",
    "n_layer": "Transformer-Blöcke", "batch_size": "Batch-Größe", "max_iters": "Trainingsschritte (eingestellt)",
    "learning_rate": "Lernrate", "dropout": "Dropout",
}


def show_model_info(name, checkpoint, model):
    config, history = checkpoint["config"], checkpoint["history"]
    n_params = sum(p.numel() for p in model.parameters())
    rows = "".join(f"<tr><td>{label}</td><td><b>{config[key]}</b></td></tr>"
                   for key, label in CONFIG_LABELS.items() if key in config)
    display(HTML(
        f"<h4>{html.escape(name)}</h4><ul>"
        f"<li><b>{_fmt(n_params)}</b> Parameter</li>"
        f"<li><b>{len(checkpoint['chars'])}</b> Zeichen im Vokabular</li></ul>"
        f"<table>{rows}</table>"
        f"<b>Vokabular:</b><pre style='background:#f6f8fa;color:#222;padding:8px;white-space:pre-wrap'>"
        f"{html.escape(''.join(_visible(c) for c in checkpoint['chars']))}</pre>"))
    if not history.get("step"):
        print("Zu diesem Modell wurde kein Trainingsverlauf gespeichert.")
        return
    baseline = math.log(len(checkpoint["chars"]))
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(history["step"], history["train"], "o-", label="Trainings-Loss")
    ax.plot(history["step"], history["val"], "o-", label="Validierungs-Loss")
    ax.axhline(baseline, ls="--", color="gray", label=f"Zufälliges Raten ({baseline:.2f})")
    ax.set_xlabel("Trainingsschritt")
    ax.set_ylabel("Loss  (kleiner = besser)")
    ax.set_ylim(0, max(baseline, max(history["train"] + history["val"])) * 1.1)
    ax.set_title(f"Trainingsverlauf: {history['step'][-1]} Schritte · Endstand Training "
                 f"{history['train'][-1]:.3f}, Validierung {history['val'][-1]:.3f}")
    ax.grid(alpha=0.3)
    ax.legend(loc="upper right")
    plt.show()


print("Hilfsfunktionen geladen. Modelle werden in", DATA_DIR, "gesucht.")


### Modell-Bauplan

Die Datei enthält nur die gelernten **Gewichte** (viele Zahlen) und die Einstellungen – nicht den Code des Modells. Darum brauchen wir hier denselben Bauplan wie in `02_mini_gpt.ipynb`. Die Gewichte werden dann in diesen Bauplan „eingesetzt“.

In [ ]:
class Head(nn.Module):
    """Ein einzelner Self-Attention-Kopf."""

    def __init__(self, n_embd, head_size, block_size, dropout):
        super().__init__()
        self.query = nn.Linear(n_embd, head_size, bias=False)  # "Wonach suche ich?"
        self.key = nn.Linear(n_embd, head_size, bias=False)    # "Was habe ich zu bieten?"
        self.value = nn.Linear(n_embd, head_size, bias=False)  # "Was gebe ich weiter?"
        self.register_buffer("mask", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)
        self.last_attention = None  # wird für die Visualisierung gespeichert

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.query(x), self.key(x), self.value(x)
        # Wie gut passt jede Frage (Query) zu jedem Etikett (Key)?
        scores = q @ k.transpose(-2, -1) / math.sqrt(k.shape[-1])
        # Maske: Kein Blick in die Zukunft!
        scores = scores.masked_fill(self.mask[:T, :T] == 0, float("-inf"))
        # Softmax macht daraus Gewichte, die zusammen 1 (= 100 %) ergeben
        weights = F.softmax(scores, dim=-1)
        self.last_attention = weights.detach()
        return self.dropout(weights) @ v


class MultiHeadAttention(nn.Module):
    """Mehrere Attention-Köpfe parallel, Ergebnisse werden zusammengefügt."""

    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        head_size = n_embd // n_head
        self.heads = nn.ModuleList([Head(n_embd, head_size, block_size, dropout) for _ in range(n_head)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([head(x) for head in self.heads], dim=-1)
        return self.dropout(self.proj(out))


class FeedForward(nn.Module):
    """Ein kleines neuronales Netz, das auf jedes Zeichen einzeln angewendet wird."""

    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """Ein Transformer-Block: erst kommunizieren (Attention), dann nachdenken (Feed-Forward)."""

    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.sa = MultiHeadAttention(n_embd, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffwd = FeedForward(n_embd, dropout)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))    # "+ x" = Residual-Verbindung
        x = x + self.ffwd(self.ln2(x))
        return x


class MiniGPT(nn.Module):
    def __init__(self, vocab_size, block_size, n_embd, n_head, n_layer, dropout):
        super().__init__()
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)  # Vektor → eine Punktzahl pro Zeichen im Vokabular

    def forward(self, idx, targets=None):
        B, T = idx.shape
        positions = torch.arange(T, device=idx.device)
        x = self.token_embedding(idx) + self.position_embedding(positions)
        for block in self.blocks:
            x = block(x)
        logits = self.lm_head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.block_size:])        # nur die letzten block_size Zeichen
            probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)  # zufällig ziehen – gemäß Wahrscheinlichkeit
            idx = torch.cat([idx, next_id], dim=1)
        return idx

## 1. Modell laden

Wähle ein gespeichertes Modell aus. Die Liste zeigt alle `.pt`-Dateien im Ordner `data/` – das neueste zuerst.

In [ ]:
auswahl = ModelChooser(device)
auswahl.show()

Wenn du deine Wahl getroffen hast, führe die nächste Zelle aus. Sie baut das Modell mit den gespeicherten Einstellungen, setzt die Gewichte ein und zeigt dir den Trainingsverlauf:

In [ ]:
modell_name, checkpoint = auswahl.load()
config = checkpoint["config"]

chars = checkpoint["chars"]
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}  # string → integer
itos = {i: ch for i, ch in enumerate(chars)}  # integer → string


def encode(s):
    """Text → Liste von Token-IDs (unbekannte Zeichen werden übersprungen)."""
    return [stoi[c] for c in s if c in stoi]


def decode(ids):
    """Liste von Token-IDs → Text."""
    return "".join(itos[i] for i in ids)


model = MiniGPT(
    vocab_size=vocab_size,
    block_size=config["block_size"],
    n_embd=config["n_embd"],
    n_head=config["n_head"],
    n_layer=config["n_layer"],
    dropout=config["dropout"],
).to(device)
model.load_state_dict(checkpoint["model"])  # die gelernten Gewichte einsetzen
model.eval()

show_model_info(modell_name, checkpoint, model)


> **Aufgabe 1:** Schau dir die Einstellungen und den Trainingsverlauf an. Wie groß ist das Modell? Wie weit ist der Loss unter „Zufälliges Raten“ gefallen? Liegen Trainings- und Validierungs-Loss nah beieinander – oder hat das Modell den Text auswendig gelernt (Overfitting)?

**Deine Antwort:**

_…hier schreiben (Doppelklick auf diese Zelle)…_

## 2. Text generieren

Jetzt darf dein Modell schreiben! Gib einen Anfang vor (gelb markiert) und lass es weiterschreiben.

**Die Temperatur** bestimmt, wie „mutig“ das Modell ist:
- **niedrig (0.1 – 0.5)** → wählt fast immer das wahrscheinlichste Zeichen: sicher, aber oft langweilig und wiederholend
- **mittel (0.7 – 1.0)** → ausgewogen
- **hoch (1.5 – 2.0)** → wählt auch unwahrscheinliche Zeichen: kreativ, aber chaotisch

In [ ]:
def generate_text(prompt="", length=300, temperature=1.0):
    """Lässt das Modell nach `prompt` weitere `length` Zeichen schreiben."""
    ids = encode(prompt) or [stoi.get("\n", 0)]
    idx = torch.tensor([ids], dtype=torch.long, device=device)
    model.eval()
    out = model.generate(idx, length, temperature)
    model.train()
    return decode(out[0].tolist())


# Ein Beispieltext – sein Anfang dient unten als Vorschlag für die Eingabefelder.
beispiel = generate_text(length=300, temperature=0.8)
print(beispiel)
start = beispiel.strip()[:30]

In [ ]:
generation_playground(generate_text, encode, decode, default_prompt=start)

### Was denkt das Modell?

Das Modell „schreibt“ nicht wirklich – es berechnet für **jedes** Zeichen im Vokabular eine Wahrscheinlichkeit und **würfelt** dann. Hier siehst du diese Wahrscheinlichkeiten. Ändere den Text und die Temperatur (mit Enter bestätigen):

In [ ]:
next_char_explorer(model, encode, itos, default_prompt=start)

> **Aufgabe 2:** Generiere Texte mit Temperatur **0.2**, **1.0** und **2.0**. Beschreibe die Unterschiede. Schau dir im Diagramm an, was die Temperatur mit den Wahrscheinlichkeiten macht.
>
> **Aufgabe 3:** Finde einen Textanfang, bei dem sich das Modell **sehr sicher** ist (über 90 %), und einen, bei dem es **unsicher** ist. Woran liegt das?

**Deine Antwort:**

_…hier schreiben (Doppelklick auf diese Zelle)…_

## 3. Ins Modell hineinschauen

### Attention: Worauf achtet das Modell?

Jede **Zeile** ist ein Zeichen. Die Farbe zeigt, wie stark dieses Zeichen auf die früheren Zeichen (Spalten) „schaut“. Dunkelblau = viel Aufmerksamkeit.
Das leere Dreieck rechts oben ist die **Maske**: kein Blick in die Zukunft.

In [ ]:
attention_explorer(model, encode, itos, default_text=start)

> **Aufgabe 4:** Wähle verschiedene Blöcke und Köpfe aus (oder „Alle Köpfe“).
> - Findest du einen Kopf, der fast nur auf das **direkt vorherige** Zeichen schaut?
> - Findest du einen Kopf, der auf **Leerzeichen** oder **Wortanfänge** achtet?

**Deine Antwort:**

_…hier schreiben (Doppelklick auf diese Zelle)…_

### Embeddings: Was hat das Modell über Zeichen gelernt?

Jedes Zeichen wird im Modell durch einen Vektor mit vielen Zahlen dargestellt. Wir quetschen diese Vektoren auf 2 Dimensionen zusammen, damit wir sie zeichnen können. Zeichen, die das Modell als **ähnlich** ansieht, liegen nahe beieinander.

In [ ]:
plot_embeddings(model, itos)

> **Aufgabe 5:** Welche Zeichen liegen nahe beieinander? Bilden Vokale, Großbuchstaben, Ziffern oder Satzzeichen Gruppen?
> Bedenke: Niemand hat dem Modell gesagt, was ein Vokal ist. Woher „weiß“ es das?

**Deine Antwort:**

_…hier schreiben (Doppelklick auf diese Zelle)…_

## 4. Modelle vergleichen

Lade in Schritt 1 ein **anderes** Modell (Zelle mit der Auswahl erneut ausführen, Modell wählen, dann alle Zellen darunter erneut ausführen) und vergleiche:

- **Anderer Datensatz:** Wie unterscheiden sich die Texte? Sehen die Embeddings anders aus – z. B. bei einem deutschen und einem englischen Text?
- **Andere Modellgröße:** Schreibt das größere Modell besser? Findest du in den größeren Modellen „spezialisiertere“ Attention-Köpfe?
- **Gleicher Anfang, andere Modelle:** Gib allen Modellen denselben Textanfang. Was macht jedes daraus?

**Deine Antwort:**

_…hier schreiben (Doppelklick auf diese Zelle)…_